In [30]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()



True

In [31]:
llm = ChatOpenAI(model='gpt-5-mini',temperature=0)

# ***Implement Rag with Text data***

In [32]:

from langchain_core.documents import Document
from importlib.metadata import metadata

my_text=""""AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation).
Part of a series on
Artificial intelligence (AI)

Major goals
Approaches
Applications
Philosophy
History
Controversies
Glossary
vte
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of artificial intelligence include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as such: "A lot of cutting-edge AI has filtered into general applications, often without being called AI because once something becomes useful enough and common enough it's not labeled AI anymore."[2][3]

Various subfields of AI research are centered around particular goals and the use of particular tools. The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers have adapted and integrated a wide range of techniques, including search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[4] Some companies, such as OpenAI, Google DeepMind and Meta,[5] aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human.

Artificial intelligence was founded as an academic discipline in 1956,[6] and the field went through multiple cycles of optimism throughout its history,[7][8] followed by periods of disappointment and loss of funding, known as AI winters.[9][10] Funding and interest increased substantially after 2012, when graphics processing units began being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[11] This growth accelerated further after 2017 with the transformer architecture.[12] In the 2020s, an ongoing period of rapid progress in advanced generative AI became known as the AI boom. Generative AI's ability to create and modify content has led to several unintended consequences and harms. Ethical concerns have been raised about AI's long-term effects and potential existential risks, prompting discussions about regulatory policies to ensure the safety and benefits of the technology.
"""


docs= [Document(page_content=my_text, metadata={"source":"https://en.wikipedia.org/wiki/Artificial_intelligence","Author":"wikipedia"})]

### **Preparing chunks**

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap = 50
)

chunks = splitter.split_documents(docs)
chunks


[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'Author': 'wikipedia'}, page_content='"AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation).\nPart of a series on\nArtificial intelligence (AI)'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'Author': 'wikipedia'}, page_content='Major goals\nApproaches\nApplications\nPhilosophy\nHistory\nControversies\nGlossary\nvte'),
 Document(metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'Author': 'wikipedia'}, page_content='Controversies\nGlossary\nvte\nArtificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive th

### **Create Embedding**

In [34]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings()


In [35]:
res=embedding_model.embed_query("cricket")

In [36]:
len(res)

1536

### **Store in Vector Store**

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(),
    collection_name="my_rag_collection"  
)


In [46]:
query = "When AI invented at first"
context=vector_store.similarity_search(query, k=3)

In [47]:
context

[Document(id='69ab95b8-3f27-48fd-9bae-b4ff750dabc1', metadata={'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'Author': 'wikipedia'}, page_content='Artificial intelligence was founded as an academic discipline in 1956,[6] and the field went through multiple cycles of optimism throughout its history,[7][8] followed by periods of disappointment and loss of funding, known as AI winters.[9][10] Funding and interest increased substantially after 2012, when graphics processing units began being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[11] This growth accelerated further after 2017 with the'),
 Document(id='8668c60c-9ad4-485e-b938-67eeff64cda4', metadata={'Author': 'wikipedia', 'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence'}, page_content='"AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation).\nPart of a series on\nArtificial intelligence (AI)'),
 Docume

In [54]:
from langchain_core.prompts import ChatPromptTemplate    

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question based only on the context below."),
    ("human", "Context: {context}\n\nQuestion: {query}")
])

chain = prompt | llm

In [56]:
response = chain.invoke({"query":query,"context":context})

In [57]:
response

AIMessage(content='Artificial intelligence was founded as an academic discipline in 1956.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 150, 'prompt_tokens': 364, 'total_tokens': 514, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DPAOxe4e35Hw0hTaxRdxiUn3dj1tq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d3fb8-9346-70f3-b088-ab07bcc29d53-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 364, 'output_tokens': 150, 'total_tokens': 514, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}})

In [58]:
response.content

'Artificial intelligence was founded as an academic discipline in 1956.'